# Day 1 — Notebook 04  
# Embeddings, Similarity, Ranking & Reranking — Made Visible

**Hands-on outcome:** Turn chunks into vectors, understand cosine similarity, inspect ranking, compare chunk strategies, and visibly show how reranking changes retrieval order.

### Core visual outputs
- Embedding summary.
- Pairwise semantic-similarity table.
- Ranked retrieval table with **distance + similarity**.
- Before/after metadata-filter comparison.
- Before/after **reranking with rank movement**.

## 1. Azure + vector store setup

In [ ]:
from pathlib import Path
import os, json, re, time, math
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

if not all([endpoint, api_key, model, embedding_model]):
    raise ValueError(
        "Missing Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL and "
        "AZURE_OPENAI_EMBEDDING_MODEL in .env"
    )

client = OpenAI(base_url=endpoint, api_key=api_key)

env_df = pd.DataFrame([
    {"Component":"Chat / Generation", "Azure deployment":model, "Purpose":"Generate grounded answers / rerank"},
    {"Component":"Embeddings", "Azure deployment":embedding_model, "Purpose":"Convert text and queries into vectors"}
])
display(env_df)

import chromadb

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

records_by_strategy = {
    s:load_jsonl(ARTIFACT_DIR / f"policy_chunks_{s}.jsonl")
    for s in ["fixed","sentence","recursive","section"]
}

corpus_df = pd.DataFrame([
    {"Strategy":s, "Chunks":len(rows), "Avg chars":round(np.mean([len(x["text"]) for x in rows]),1)}
    for s,rows in records_by_strategy.items()
])
display(corpus_df)

## 2. What does an embedding actually look like?

In [ ]:
sample_texts = [
    "MRI scans require prior authorization.",
    "Advanced imaging needs insurer approval.",
    "The member updated a mailing address."
]

resp = client.embeddings.create(model=embedding_model, input=sample_texts)
sample_vectors = [x.embedding for x in resp.data]

embedding_df = pd.DataFrame([
    {
        "Text":text,
        "Vector Dimensions":len(vec),
        "First 6 Values":str([round(v,4) for v in vec[:6]]),
        "Vector Norm":round(float(np.linalg.norm(vec)),4)
    }
    for text,vec in zip(sample_texts,sample_vectors)
])
display(embedding_df)

An embedding is a long numeric vector representing semantic characteristics of the text. We normally do **not** interpret individual dimensions; we compare vectors geometrically.

### Cosine similarity
- **1.0** → vectors point in almost the same direction.
- **0.0** → weak directional relationship.
- **Negative values** → opposing directions are mathematically possible.

For a Chroma collection configured with cosine distance:

**cosine similarity ≈ 1 − cosine distance**

So in the retrieval tables below:
- **lower distance is better**
- **higher similarity is better**

## 3. Make semantic similarity visible with a small comparison matrix

In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    (0,1,"Same business meaning, different wording"),
    (0,2,"Different business meaning"),
    (1,2,"Different business meaning")
]

sim_rows = []
for i,j,relationship in pairs:
    sim_rows.append({
        "Text A":sample_texts[i],
        "Text B":sample_texts[j],
        "Expected Relationship":relationship,
        "Cosine Similarity":round(cosine_similarity(sample_vectors[i],sample_vectors[j]),4)
    })

display(pd.DataFrame(sim_rows))

> **** Embeddings let “insurer approval” retrieve “prior authorization” even though the exact terms differ.

## 4. Embed all chunk strategies and persist them in Chroma

In [ ]:
def embed_texts(texts, batch_size=32):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        vectors.extend([row.embedding for row in response.data])
    return vectors

VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_policy_db")
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

def reset_collection(name):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    return chroma_client.create_collection(name=name, metadata={"hnsw:space":"cosine"})

collections = {}

for strategy, records in records_by_strategy.items():
    vectors = embed_texts([r["text"] for r in records])
    collection = reset_collection(f"policy_{strategy}")

    collection.upsert(
        ids=[r["chunk_id"] for r in records],
        documents=[r["text"] for r in records],
        embeddings=vectors,
        metadatas=[{
            "doc_id":r["doc_id"],
            "source_file":r["source_file"],
            "title":r["title"],
            "plan_type":r["plan_type"],
            "policy_domain":r["policy_domain"],
            "effective_date":r["effective_date"],
            "page":r["page"],
            "section":r["section"],
            "chunk_strategy":r["chunk_strategy"]
        } for r in records]
    )
    collections[strategy] = collection

vector_store_df = pd.DataFrame([
    {"Collection":f"policy_{s}", "Strategy":s, "Vectors Stored":c.count()}
    for s,c in collections.items()
])
display(vector_store_df)

## 5. Ranked semantic retrieval — show rank, distance, similarity, source and text

In [ ]:
def semantic_search(question, collection, top_k=5, where=None):
    qvec = client.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    return collection.query(
        query_embeddings=[qvec],
        n_results=top_k,
        where=where,
        include=["documents","metadatas","distances"]
    )

def search_to_df(result):
    rows = []
    for rank,(doc,meta,distance) in enumerate(
        zip(result["documents"][0],result["metadatas"][0],result["distances"][0]), start=1
    ):
        rows.append({
            "Rank":rank,
            "Cosine Distance ↓":round(distance,4),
            "Cosine Similarity ↑":round(1-distance,4),
            "Document":meta["doc_id"],
            "Plan":meta["plan_type"],
            "Section":meta["section"],
            "Page":meta["page"],
            "Chunk Strategy":meta["chunk_strategy"],
            "Retrieved Text":re.sub(r"\s+"," ",doc)[:240] + "..."
        })
    return pd.DataFrame(rows)

question = "For Gold PPO, when does physical therapy start requiring authorization?"
result = semantic_search(question, collections["section"], top_k=5)
display(search_to_df(result))

### Reading this ranking

**Rank 1** is the nearest vector result.  
The similarity score is useful for comparison **within the same embedding model and use case**, but it is not a universal confidence percentage.

> **Important:** A high vector similarity means “semantically close,” not “factually correct.”

## 6. Metadata filtering — show before and after in one table

In [ ]:
query = "When does physical therapy require authorization?"

before = search_to_df(semantic_search(query, collections["section"], top_k=5))
after = search_to_df(
    semantic_search(
        query,
        collections["section"],
        top_k=5,
        where={"plan_type":"Silver HMO"}
    )
)

before["Search Mode"] = "Semantic only"
after["Search Mode"] = "Semantic + Silver HMO filter"

display(
    pd.concat([before,after], ignore_index=True)[
        ["Search Mode","Rank","Cosine Similarity ↑","Document","Plan","Section","Retrieved Text"]
    ]
)

> **Takeaway:** Filtering changes the candidate universe *before* ranking. It stops the search engine from ranking the wrong plan even when its text is highly similar.

## 7. Compare retrieval across four chunking strategies

In [ ]:
query = "What information is needed for continued physical therapy authorization?"

strategy_results = []
for strategy, collection in collections.items():
    df = search_to_df(semantic_search(query, collection, top_k=3))
    for _,row in df.iterrows():
        strategy_results.append({
            "Strategy":strategy,
            "Rank":row["Rank"],
            "Similarity":row["Cosine Similarity ↑"],
            "Document":row["Document"],
            "Section":row["Section"],
            "Text":row["Retrieved Text"]
        })

display(pd.DataFrame(strategy_results))

This output is deliberately comparative: the question is unchanged; only chunk boundaries change.

> **Teaching question:** “Which chunking method puts the complete business rule closest to the top?”

## 8. Ranking vs reranking

**Initial ranking** uses vector similarity and is fast.  
**Reranking** takes the top candidate chunks and evaluates their relevance more deeply against the actual question.

For training, we use the Azure chat model as a simple cross-encoder-style reranker:
1. Retrieve top 6 by vector similarity.
2. Ask the LLM to score each candidate from 0–100 for direct answer relevance.
3. Sort by rerank score.
4. Compare rank movement.

In [ ]:
def rerank_with_llm(question, initial_df):
    candidates = []
    for _,row in initial_df.iterrows():
        candidates.append({
            "rank":int(row["Rank"]),
            "chunk_id":f"C{int(row['Rank'])}",
            "document":row["Document"],
            "section":row["Section"],
            "text":row["Retrieved Text"]
        })

    prompt = f"""
You are reranking retrieved healthcare policy chunks for a user question.

QUESTION:
{question}

CANDIDATES:
{json.dumps(candidates, indent=2)}

Score every candidate from 0 to 100 for how directly it contains evidence needed to answer the question.
Return ONLY a JSON array in this format:
[
  {{"chunk_id":"C1","relevance_score":95,"reason":"..."}}
]
Do not omit any candidate.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You are a precise enterprise search reranker."},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )

    text = response.choices[0].message.content.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.I|re.S)
    scores = json.loads(text)
    score_map = {x["chunk_id"]:x for x in scores}

    reranked = initial_df.copy()
    reranked["Candidate"] = [f"C{x}" for x in reranked["Rank"]]
    reranked["Rerank Score ↑"] = reranked["Candidate"].map(
        lambda x:score_map.get(x,{}).get("relevance_score",0)
    )
    reranked["Rerank Reason"] = reranked["Candidate"].map(
        lambda x:score_map.get(x,{}).get("reason","")
    )

    reranked = reranked.sort_values(
        ["Rerank Score ↑","Cosine Similarity ↑"], ascending=[False,False]
    ).reset_index(drop=True)

    reranked["Rank After Rerank"] = np.arange(1,len(reranked)+1)
    reranked["Rank Movement"] = reranked["Rank"] - reranked["Rank After Rerank"]
    return reranked

rerank_question = "What clinical documentation is required when requesting continued physical therapy?"
initial = search_to_df(
    semantic_search(rerank_question, collections["section"], top_k=6)
)

display(
    initial[
        ["Rank","Cosine Similarity ↑","Document","Section","Retrieved Text"]
    ].rename(columns={"Rank":"Vector Rank"})
)

In [ ]:
reranked = rerank_with_llm(rerank_question, initial)

display(
    reranked[
        ["Rank","Rank After Rerank","Rank Movement","Cosine Similarity ↑",
         "Rerank Score ↑","Document","Section","Rerank Reason"]
    ].rename(columns={"Rank":"Vector Rank"})
)

### The reranking output

- **Vector Rank** = broad semantic closeness.
- **Rerank Score** = deeper question-to-chunk relevance assessment.
- **Rank Movement > 0** = chunk moved upward after reranking.
- **Rank Movement < 0** = chunk moved downward.

> **Takeaway:** Retrieval finds candidates; reranking decides which candidates deserve the scarce top context positions.

## 9. Top-k — convert the trade-off into a visible summary

In [ ]:
rows = []
query = "Gold PPO chiropractic annual visit limit"

for k in [1,3,5,8]:
    result = semantic_search(query, collections["section"], top_k=k)
    df = search_to_df(result)
    rows.append({
        "Top-K":k,
        "Best Similarity":df["Cosine Similarity ↑"].max(),
        "Unique Documents":df["Document"].nunique(),
        "Unique Plans":df["Plan"].nunique(),
        "Retrieved Characters":sum(len(x) for x in result["documents"][0]),
        "Top Result":f"{df.iloc[0]['Document']} | {df.iloc[0]['Section']}"
    })

display(pd.DataFrame(rows))

> **Takeaway:** Top-k is a recall-versus-noise control. Higher k increases the evidence pool but also increases downstream context, token cost and distraction.

## Day 1 completion

You can now explain the complete retrieval mechanics visually:

**Chunk strategy → embedding → cosine similarity → initial ranking → metadata filtering → reranking → top-k**

Day 2 will show how these ranked chunks become grounded answers — and what happens when the evidence is missing.